In [1]:
import json, pandas as pd
from pathlib import Path
R=Path('results') if Path('results').exists() else Path('experiments/E12B_gpt_prompt_optimization/results')
d=json.load(open(R/'e12b_analysis.json')); S=d['summary']; K=['gpt_p0','gpt_p1','gpt_p2','gpt_p3']
pd.set_option('display.width',220,'display.max_colwidth',80)

# E12B — GPT-specific prompt optimisation (Stage B)
## 1. Research question
Can a compact GPT-specific classification prompt materially improve evidence-grounded NDA classification over the Qwen-selected P0 without changing model, retrieval, context size, schema, or evaluator?

## 2. Why GPT-specific optimisation
P0 was selected on Qwen2.5-7B (E03). GPT-5-mini is far stronger (E08B); P0 was never optimised for it.

## 3. Fresh manifest / contamination controls
TRAIN_GPT_PROMPT_v1: 150 cases (50/50/50), 74 docs, seed 1200, document-disjoint from TRAIN_PROMPT_v1, TRAIN_ARCH_v1 and TRAIN_ORACLE_v1 (union 227 docs; 196 untouched).

In [2]:
print(json.load(open(R/'manifest_inspection.json'))['inspection'])

{'total_train_docs': 423, 'total_train_cases': 7191, 'docs_TRAIN_PROMPT_v1': 73, 'docs_TRAIN_ARCH_v1': 78, 'overlap_PROMPT_ARCH_docs': 19, 'overlap_PROMPT_ORACLE_docs': 29, 'overlap_ARCH_ORACLE_docs': 29, 'union_of_three_prior_manifests_docs': 227, 'docs_in_either': 227, 'untouched_docs': 196, 'untouched_label_distribution_cases': {'NotMentioned': 1339, 'Entailment': 1611, 'Contradiction': 382}, 'docs_TRAIN_ORACLE_v1': 145}


## 4. Frozen retrieval/model setup
retrieval_v1 top-5 (one shared context artifact), openai/gpt-5-mini via OpenRouter, temperature 0.0, 60s timeout for all arms, unchanged parser/validator/scorer. Schema quirk (label-only schema block + trailing evidence request) is preserved in all four arms.

## 5-6. Prompt definitions and length

In [3]:
f=json.load(open(R/'pre_run_forecast.json'))['prompt_stats']; pd.DataFrame(f).T

,chars,tokens_cl100k,added_instructions_vs_P0,diff_added_lines,diff_removed_lines
gpt_p0,464,118,0,0,0
gpt_p1,698,154,3,3,3
gpt_p2,1121,242,9,11,3
gpt_p3,1158,248,7,9,3


## 7. Execution integrity
P0 ran sequentially (150/150). A provider connection outage interrupted P1 after 5 failed attempts ($0, excluded, preserved). P1 (resume), P2, P3 then ran with bounded concurrency = 5 within each arm, arms sequential. Per-call latency is comparable; total wall-clock is not.

In [4]:
print(json.load(open(R/'pre_run_verification_resume.json'))['checks']); print(json.dumps(d['run'],indent=1)[:1500]); print(json.load(open(R/'P1_INTERRUPTED_PROVIDER_OUTAGE_20260926T171526Z.meta.json'))['note'])

{'manifest_seed_1200_150': True, 'context_same_ids_order': True, 'context_top5': True, 'context_no_gold_leakage': True, 'prompt_hashes_match_stageA': True, 'p0_complete_150_order_hash_no_errors': True, 'outage_artifact_preserved': True, 'no_existing_resume_outputs': True, 'budget_gate_pass': True}
{
 "arms": {
  "gpt_p1": {
   "spend_usd": 0.2635749999999999,
   "wall_seconds": 228.41291229100898,
   "rate_limit_like_errors": 0,
   "stopped": null,
   "failed_calls": 0,
   "verify": {
    "records_150": true,
    "no_duplicates": true,
    "all_case_ids_present": true,
    "all_successful": true,
    "prompt_hash_ok": true,
    "run_id_ok": true
   }
  },
  "gpt_p2": {
   "spend_usd": 0.28801949999999993,
   "wall_seconds": 237.61332141701132,
   "rate_limit_like_errors": 0,
   "stopped": null,
   "failed_calls": 0,
   "verify": {
    "records_150": true,
    "no_duplicates": true,
    "all_case_ids_present": true,
    "all_successful": true,
    "prompt_hash_ok": true,
    "run_id_ok"

## 8. GPT-P0 metrics

In [5]:
s=S['gpt_p0']; print({x:round(s[x],4) for x in ['accuracy','macro_f1','evidence_recall','evidence_precision','source_valid_evidence_rate','joint']}); print('recall',s['recall'],'joint_by_class',s['joint_by_class']); print('parse',s['parse'],'errors',s['errors']); print('confusion (rows=gold E,C,NM)',s['confusion_matrix'])

{'accuracy': 0.8133, 'macro_f1': 0.8136, 'evidence_recall': 0.85, 'evidence_precision': 0.8252, 'source_valid_evidence_rate': 0.9709, 'joint': 0.76}
recall {'Entailment': 0.82, 'Contradiction': 0.82, 'NotMentioned': 0.8} joint_by_class {'Entailment': 0.68, 'Contradiction': 0.8, 'NotMentioned': 0.8}
parse {'strict': 150, 'recovered': 0, 'invalid': 0} errors 0
confusion (rows=gold E,C,NM) [[41, 3, 6], [8, 41, 1], [4, 6, 40]]


## 9. GPT-P1 metrics

In [6]:
s=S['gpt_p1']; print({x:round(s[x],4) for x in ['accuracy','macro_f1','evidence_recall','evidence_precision','source_valid_evidence_rate','joint']}); print('recall',s['recall'],'joint_by_class',s['joint_by_class']); print('parse',s['parse'],'errors',s['errors']); print('confusion (rows=gold E,C,NM)',s['confusion_matrix'])

{'accuracy': 0.7867, 'macro_f1': 0.7867, 'evidence_recall': 0.85, 'evidence_precision': 0.8019, 'source_valid_evidence_rate': 0.9717, 'joint': 0.7467}
recall {'Entailment': 0.78, 'Contradiction': 0.84, 'NotMentioned': 0.74} joint_by_class {'Entailment': 0.68, 'Contradiction': 0.82, 'NotMentioned': 0.74}
parse {'strict': 150, 'recovered': 0, 'invalid': 0} errors 0
confusion (rows=gold E,C,NM) [[39, 6, 5], [6, 42, 2], [8, 5, 37]]


## 10. GPT-P2 metrics

In [7]:
s=S['gpt_p2']; print({x:round(s[x],4) for x in ['accuracy','macro_f1','evidence_recall','evidence_precision','source_valid_evidence_rate','joint']}); print('recall',s['recall'],'joint_by_class',s['joint_by_class']); print('parse',s['parse'],'errors',s['errors']); print('confusion (rows=gold E,C,NM)',s['confusion_matrix'])

{'accuracy': 0.76, 'macro_f1': 0.7608, 'evidence_recall': 0.85, 'evidence_precision': 0.787, 'source_valid_evidence_rate': 0.963, 'joint': 0.7067}
recall {'Entailment': 0.76, 'Contradiction': 0.8, 'NotMentioned': 0.72} joint_by_class {'Entailment': 0.62, 'Contradiction': 0.78, 'NotMentioned': 0.72}
parse {'strict': 150, 'recovered': 0, 'invalid': 0} errors 0
confusion (rows=gold E,C,NM) [[38, 8, 4], [8, 40, 2], [7, 7, 36]]


## 11. GPT-P3 metrics

In [8]:
s=S['gpt_p3']; print({x:round(s[x],4) for x in ['accuracy','macro_f1','evidence_recall','evidence_precision','source_valid_evidence_rate','joint']}); print('recall',s['recall'],'joint_by_class',s['joint_by_class']); print('parse',s['parse'],'errors',s['errors']); print('confusion (rows=gold E,C,NM)',s['confusion_matrix'])

{'accuracy': 0.84, 'macro_f1': 0.8403, 'evidence_recall': 0.92, 'evidence_precision': 0.8846, 'source_valid_evidence_rate': 0.9904, 'joint': 0.8133}
recall {'Entailment': 0.9, 'Contradiction': 0.82, 'NotMentioned': 0.8} joint_by_class {'Entailment': 0.82, 'Contradiction': 0.82, 'NotMentioned': 0.8}
parse {'strict': 150, 'recovered': 0, 'invalid': 0} errors 0
confusion (rows=gold E,C,NM) [[45, 1, 4], [7, 41, 2], [6, 4, 40]]


## 12. Four-prompt comparison

In [9]:
rows={'Accuracy':'accuracy','Macro-F1':'macro_f1','Evidence recall':'evidence_recall','Evidence precision':'evidence_precision','Source-valid evidence':'source_valid_evidence_rate','Joint':'joint'}
t=pd.DataFrame({k:{n:S[k][m] for n,m in rows.items()} for k in K})
for c in ['Entailment','Contradiction','NotMentioned']: t.loc[c+' recall']=[S[k]['recall'][c] for k in K]
for c in ['Entailment','Contradiction','NotMentioned']: t.loc['Joint '+c]=[S[k]['joint_by_class'][c] for k in K]
t.loc['Joint (cases /150)']=[S[k]['joint_n'] for k in K]; t.round(4)

,gpt_p0,gpt_p1,gpt_p2,gpt_p3
Accuracy,0.8133,0.7867,0.7600,0.8400
Macro-F1,0.8136,0.7867,0.7608,0.8403
Evidence recall,0.8500,0.8500,0.8500,0.9200
Evidence precision,0.8252,0.8019,0.7870,0.8846
Source-valid evidence,0.9709,0.9717,0.9630,0.9904
Joint,0.7600,0.7467,0.7067,0.8133
Entailment recall,0.8200,0.7800,0.7600,0.9000
Contradiction recall,0.8200,0.8400,0.8000,0.8200
NotMentioned recall,0.8000,0.7400,0.7200,0.8000
Joint Entailment,0.6800,0.6800,0.6200,0.8200


## 13. Contradiction analysis (E03 concern: richer prompts pushing Contradiction → NotMentioned)

In [10]:
print(pd.DataFrame(d['contradiction']).T); pd.DataFrame([{'case_id':r['case_id'],**{k[-2:].upper():v for k,v in r['labels'].items()}} for r in d['contradiction_prompt_sensitive']])

        correct    of  recall  to_Entailment  to_NotMentioned  joint
gpt_p0     41.0  50.0    0.82            8.0              1.0   0.80
gpt_p1     42.0  50.0    0.84            6.0              2.0   0.82
gpt_p2     40.0  50.0    0.80            8.0              2.0   0.78
gpt_p3     41.0  50.0    0.82            7.0              2.0   0.82


,case_id,P0,P1,P2,P3
0,train::87::nda-7,Entailment,Contradiction,Entailment,Entailment
1,train::111::nda-20,Contradiction,Entailment,Entailment,Contradiction
2,train::118::nda-7,Contradiction,Contradiction,NotMentioned,NotMentioned
3,train::176::nda-20,Entailment,Contradiction,Entailment,Entailment
4,train::176::nda-7,Contradiction,Entailment,Contradiction,Entailment
5,train::219::nda-7,Contradiction,NotMentioned,Contradiction,Contradiction
6,train::441::nda-20,Entailment,Contradiction,Contradiction,Contradiction
7,train::475::nda-17,Entailment,Contradiction,Entailment,Contradiction


## 14-15. Classification and joint transitions vs P0 (with discordant case IDs)

In [11]:
for k,v in d['paired_vs_p0'].items():
    print(k,'CLS',v['cls'],'JOINT',v['joint']); print('  joint fail->success',v['joint_fail_to_success_ids']); print('  joint success->fail',v['joint_success_to_fail_ids']); print('  cls wrong->right',v['cls_wrong_to_right_ids']); print('  cls right->wrong',v['cls_right_to_wrong_ids'])

gpt_p1 CLS {'wrong_to_right': 10, 'right_to_wrong': 14, 'both_right': 108, 'both_wrong': 18} JOINT {'wrong_to_right': 12, 'right_to_wrong': 14, 'both_right': 100, 'both_wrong': 24}
  joint fail->success ['train::87::nda-7', 'train::176::nda-20', 'train::441::nda-20', 'train::475::nda-17', 'train::183::nda-12', 'train::183::nda-13', 'train::191::nda-15', 'train::255::nda-15', 'train::604::nda-11', 'train::415::nda-1', 'train::489::nda-1', 'train::512::nda-1']
  joint success->fail ['train::111::nda-20', 'train::176::nda-7', 'train::219::nda-7', 'train::255::nda-16', 'train::442::nda-12', 'train::448::nda-1', 'train::562::nda-10', 'train::581::nda-10', 'train::123::nda-1', 'train::255::nda-10', 'train::255::nda-11', 'train::308::nda-11', 'train::562::nda-15', 'train::621::nda-11']
  cls wrong->right ['train::87::nda-7', 'train::176::nda-20', 'train::441::nda-20', 'train::475::nda-17', 'train::191::nda-15', 'train::255::nda-15', 'train::604::nda-11', 'train::415::nda-1', 'train::489::nda-

## 16. Prompt-sensitive cases

In [12]:
s=d['sensitivity']; print({k:s[k] for k in ['all_correct','all_wrong','prompt_sensitive','families','evidence_drift_cases']}); pd.DataFrame(s['rows'])[['case_id','gold','pattern','evidence_drift_across_prompts']]

{'all_correct': 102, 'all_wrong': 14, 'prompt_sensitive': 34, 'families': {'C<->E': 18, 'C<->NM': 13, 'E<->NM': 9}, 'evidence_drift_cases': 33}


,case_id,gold,pattern,evidence_drift_across_prompts
0,train::87::nda-7,Contradiction,P0=E P1=C P2=E P3=E,True
1,train::111::nda-20,Contradiction,P0=C P1=E P2=E P3=C,True
2,train::118::nda-7,Contradiction,P0=C P1=C P2=NM P3=NM,True
3,train::176::nda-20,Contradiction,P0=E P1=C P2=E P3=E,True
4,train::176::nda-7,Contradiction,P0=C P1=E P2=C P3=E,True
5,train::219::nda-7,Contradiction,P0=C P1=NM P2=C P3=C,True
6,train::441::nda-20,Contradiction,P0=E P1=C P2=C P3=C,True
7,train::475::nda-17,Contradiction,P0=E P1=C P2=E P3=C,True
8,train::191::nda-15,Entailment,P0=C P1=E P2=E P3=E,True
9,train::196::nda-10,Entailment,P0=E P1=E P2=C P3=E,True


## 17. Evidence-sensitive cases (prompt-sensitive cases only)

In [13]:
pd.DataFrame(s['evidence_counts']).fillna(0).astype(int)

,gpt_p0,gpt_p1,gpt_p2,gpt_p3
wrong_label_source_valid_evidence,7,10,10,4
correct_label_gold_valid_evidence,19,16,11,23
wrong_label_gold_overlap_evidence,5,6,10,4
wrong_label_other,2,2,2,2
correct_label_bad_evidence,1,0,1,1
evidence_omitted,0,0,0,0


## 18. Paired statistics (supporting evidence only; not a selection gate)

In [14]:
for k,v in d['paired_vs_p0'].items():
    print(k,'McNemar cls',v['mcnemar_cls'],'| joint',v['mcnemar_joint'])
    print('  bootstrap',{a:(round(b['point'],3),[round(x,3) for x in b['ci95']]) for a,b in v['bootstrap'].items()})

gpt_p1 McNemar cls {'p0_only': 14, 'candidate_only': 10, 'n_discordant': 24, 'p': 0.5412561893463135} | joint {'p0_only': 14, 'candidate_only': 12, 'n_discordant': 26, 'p': 0.8450189828872681}
  bootstrap {'accuracy': (-0.027, [-0.093, 0.04]), 'macro_f1': (-0.027, [-0.092, 0.038]), 'contradiction_recall': (0.02, [-0.083, 0.125]), 'joint': (-0.013, [-0.08, 0.053])}
gpt_p2 McNemar cls {'p0_only': 13, 'candidate_only': 5, 'n_discordant': 18, 'p': 0.09625244140625} | joint {'p0_only': 15, 'candidate_only': 7, 'n_discordant': 22, 'p': 0.13380050659179688}
  bootstrap {'accuracy': (-0.053, [-0.107, 0.0]), 'macro_f1': (-0.053, [-0.109, 0.001]), 'contradiction_recall': (-0.02, [-0.093, 0.044]), 'joint': (-0.053, [-0.113, 0.007])}
gpt_p3 McNemar cls {'p0_only': 7, 'candidate_only': 11, 'n_discordant': 18, 'p': 0.480682373046875} | joint {'p0_only': 7, 'candidate_only': 15, 'n_discordant': 22, 'p': 0.13380050659179688}
  bootstrap {'accuracy': (0.027, [-0.027, 0.08]), 'macro_f1': (0.027, [-0.028

## 19. Cost / token / latency (per-call latency; NOT total wall-clock)

In [15]:
pd.DataFrame(d['cost_token_latency']).T.round(3)

,prompt_tokens,mean_input,mean_output,total_cost,cost_delta_vs_p0,cost_delta_pct,latency_mean_ms,latency_delta_pct,output_delta_pct
gpt_p0,118.0,1140.987,702.080,0.253,0.000,0.000,7582.086,0.000,0.000
gpt_p1,154.0,1176.987,731.460,0.264,0.010,4.011,7468.870,-1.493,4.185
gpt_p2,242.0,1263.987,802.067,0.288,0.035,13.657,7814.567,3.066,14.241
gpt_p3,248.0,1270.987,677.460,0.251,-0.003,-0.991,7093.624,-6.442,-3.507


In [16]:
pd.DataFrame({k:{'lat mean ms':S[k]['latency_ms']['mean'],'lat median':S[k]['latency_ms']['median'],'lat p90':S[k]['latency_ms']['p90'],'lat max':S[k]['latency_ms']['max'],'out mean':S[k]['output_tokens']['mean'],'out p90':S[k]['output_tokens']['p90']} for k in K}).round(0)

,gpt_p0,gpt_p1,gpt_p2,gpt_p3
lat mean ms,7582.0,7469.0,7815.0,7094.0
lat median,7137.0,7057.0,7301.0,6394.0
lat p90,11561.0,12014.0,13022.0,11494.0
lat max,35657.0,16495.0,21469.0,22454.0
out mean,702.0,731.0,802.0,677.0
out p90,1188.0,1250.0,1310.0,1135.0


## 20. Noise / repeatability caveat
E12A showed hosted-model outcomes flip on identical context. Prompt-sensitive cases here (34/150) include flips that need not be caused by wording. No case was re-run. Effects of 1-2 cases are not interpreted.

## 21. Selection-rule application (frozen; Macro-F1 'materially worse' = drop > 0.03, fixed before results)

In [17]:
pd.DataFrame(d['selection_rule']).T

,contradiction_delta_cases,guard_pass,net_joint_cases,macro_f1_delta,macro_f1_ok,evidence_delta_vs_p0,status
gpt_p1,1,True,-2,-0.026936,True,"{'recall': 0.0, 'precision': -0.023355925993771676}",NO BENEFIT
gpt_p2,-1,True,-8,-0.052827,False,"{'recall': 0.0, 'precision': -0.03820568140956482}",HARMFUL
gpt_p3,0,True,8,0.026658,True,"{'recall': 0.07000000000000006, 'precision': 0.05937266616878267}",NEAR-TIE


## 22. Final E12B outcome

In [18]:
print(d['outcome'])

B - EFFECTIVELY TIED (near-tie) - KEEP P0
